# T67 — Carbonate-platform degassing

**Cluster L: Mineral exploration (extension).**

Carbon emitted from continental-arc volcanoes tracks *what has been subducted*. When a subduction zone eats an oceanic plateau or a paleo-continental margin carrying a thick carbonate platform, the platform's Ca-Mg-carbonate is metamorphosed, decarbonated, and released as CO₂ from the arc above. Mather, Müller, Dutkiewicz & Zahirovic (2026, *Communications Earth & Environment* 7:48) build a full quantitative CO₂-degassing time series by threading four steps together: (1) water in oceanic lithosphere → (2) water subducted at trenches → (3) slab devolatilisation → (4) carbonate-platform arc degassing scaled by mantle melting rate.

**Scope of this notebook.** The full workflow lives in Ben Mather's [carbonate-platform-degassing](https://github.com/brmather/carbonate-platform-degassing) repository as four notebooks with `melt` + `slabdip` dependencies and a Zenodo data bundle. This tutorial ports the **first-order geometrical driver** — the map of *active carbonate platforms* through time from Zahirovic et al. (2022) — and shows the workflow's ingredients: platforms in paleo-position, their intersection with subduction zones (built on T66's SZ tessellation), and a global platform-area time series that correlates with the icehouse-greenhouse alternation Mather 2026 quantifies. The actual CO₂-flux numbers require the full four-notebook Mather workflow.

## What this notebook produces

1. **§3 — Load active-carbonate-platform features.** Ben Mather's `Zahirovic_etal_2022_mod-ActiveCarbonatePlatforms.gpml` (bundled) — polygon set with birth/death times marking when each carbonate platform was actively depositing.
2. **§4 — Paleo-Earth snapshot at one age.** Active platforms plotted as filled polygons on a Z22 paleo-Earth alongside subduction zones from T66's tessellation. Highlights platforms near active margins — the primary degassing candidates.
3. **§5 — Snapshot mosaic.** Five ages (250, 200, 150, 100, 50 Ma) showing the geographic march of carbonate platform emplacement + subduction through the Mesozoic-Cenozoic.
4. **§6 — Time series of active-platform area.** Global integrated area of active carbonate platforms through the Phanerozoic. This is the primary geometrical driver of degassing rate in the Mather 2026 workflow.

## Learning objectives

- Load a plate-topology-tied polygon feature-set from a `.gpml` file via `pygplates.FeatureCollection` and drive it through Z22 topologies.
- Reconstruct each polygon to a target age using `PlateReconstruction.reconstruct` (feature-level, not point-level).
- Compute polygon area on a spherical Earth using `pygplates.GeometryOnSphere.get_area`.
- Cross-reference platform locations with T66's subduction-zone tessellation to identify degassing-prone platforms.

## Prerequisites and runtime

- Bundled `data/carbonate_platforms/Zahirovic_etal_2022_mod-ActiveCarbonatePlatforms.gpml` (~7 MB) — pulled from Ben Mather's repo.
- Python: `gplately`, `pygplates`, `pygmt`, `pandas`, `numpy`, `matplotlib`, `geopandas`, `shapely`.
- Runtime: ~90 s (§6 loop over 40 timesteps sequentially reconstructs the polygon set at each step).


## Environment + imports


In [13]:
from pathlib import Path
import os, sys
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import pandas as pd
import geopandas as gpd
import pygmt
import gplately
import pygplates
from plate_model_manager import PlateModelManager

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, pd, gpd, pygmt, gplately, pygplates):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


Environment
  python      3.12.5
  numpy       2.3.2
  pandas      2.2.3
  geopandas   1.0.1
  pygmt       v0.18.0
  gplately    2.0.0.post19+git.2cce7bb3
  pygplates   1.0.0


In [14]:
# === USER CONFIGURATION =====================================================
MODEL_NAME              = "Zahirovic2022"
ANCHOR_PLATE_ID         = 0                    # mantle frame

# Single-snapshot analysis
RECONSTRUCTION_TIME     = 100.0                # mid-Cretaceous — peak carbonate platform activity

# Snapshot mosaic ages
SNAPSHOT_AGES_MA        = [250, 200, 150, 100, 50]

# Time-series loop
AREA_TIME_MIN_MA        = 0
AREA_TIME_MAX_MA        = 250
AREA_TIME_STEP_MA       = 5

# Carbonate-platform feature file (from brmather/carbonate-platform-degassing)
PLATFORMS_GPML          = Path("data/carbonate_platforms/"
                               "Zahirovic_etal_2022_mod-ActiveCarbonatePlatforms.gpml")

# SZ tessellation (same as T66)
TESSELLATE_KM           = 30.0

# Map
REGION, PROJ            = [-180, 180, -90, 90], "N15c"
# ============================================================================
print(f"  model:        {MODEL_NAME}")
print(f"  snapshot:     {RECONSTRUCTION_TIME} Ma")
print(f"  mosaic ages:  {SNAPSHOT_AGES_MA}")
print(f"  area loop:    {AREA_TIME_MIN_MA}-{AREA_TIME_MAX_MA} Ma at "
      f"{AREA_TIME_STEP_MA}-Myr cadence")


  model:        Zahirovic2022
  snapshot:     100.0 Ma
  mosaic ages:  [250, 200, 150, 100, 50]
  area loop:    0-250 Ma at 5-Myr cadence


## 1. Load plate model + carbonate-platform features


In [15]:
pmm   = PlateModelManager()
model = pmm.get_model(MODEL_NAME, data_dir="data/pmm_cache")

recon = gplately.PlateReconstruction(
    rotation_model=model.get_rotation_model(),
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
)
gplot = gplately.PlotTopologies(
    plate_reconstruction=recon,
    coastlines=model.get_layer("Coastlines"),
    plot_engine=gplately.PygmtPlotEngine(),
    continents=model.get_layer("ContinentalPolygons"),
    time=RECONSTRUCTION_TIME,
    anchor_plate_id=ANCHOR_PLATE_ID,
)

# Load active-carbonate-platforms as a FeatureCollection
platforms_fc = pygplates.FeatureCollection(str(PLATFORMS_GPML))
print(f"  {MODEL_NAME} loaded")
print(f"  {len(platforms_fc)} active-carbonate-platform features loaded")

# Peek at the first few features
for i, feat in enumerate(platforms_fc):
    if i >= 5: break
    valid = feat.get_valid_time()
    plate_id = feat.get_reconstruction_plate_id()
    name = feat.get_name(default="(unnamed)")
    print(f"    [{i}] {name:32s} plate {plate_id:>4d}  valid {valid[0]:>6.1f} - {valid[1]:>6.1f} Ma")


  Zahirovic2022 loaded
  730 active-carbonate-platform features loaded
    [0] (unnamed)                        plate  205  valid   12.7 -    0.0 Ma
    [1] (unnamed)                        plate  101  valid   12.7 -    0.0 Ma
    [2] (unnamed)                        plate  234  valid   12.7 -    0.0 Ma
    [3] (unnamed)                        plate  234  valid   12.7 -    0.0 Ma
    [4] (unnamed)                        plate  234  valid   12.7 -    0.0 Ma


## 2. Reconstruct platforms to the target age + render

We reconstruct each platform polygon to its position at `RECONSTRUCTION_TIME`, then rasterise via `pygplates.reconstruct` into a set of geometries that pyGMT can plot as filled polygons.


In [16]:
def reconstructed_platforms_gdf(time_ma):
    """Return a GeoDataFrame of platform polygons reconstructed to time_ma."""
    reconstructed_feats = []
    pygplates.reconstruct(platforms_fc, recon.rotation_model, reconstructed_feats,
                          float(time_ma), anchor_plate_id=ANCHOR_PLATE_ID)
    # Extract each reconstructed geometry
    rows = []
    for rf in reconstructed_feats:
        geom = rf.get_reconstructed_geometry()
        feat = rf.get_feature()
        lat_lon = geom.to_lat_lon_list() if hasattr(geom, "to_lat_lon_list") else None
        if lat_lon is None: continue
        # Convert to shapely polygon
        from shapely.geometry import Polygon
        try:
            poly = Polygon([(lon, lat) for lat, lon in lat_lon])
            if not poly.is_valid: poly = poly.buffer(0)
            rows.append({
                "name":  feat.get_name(default=""),
                "plate_id": feat.get_reconstruction_plate_id(),
                "geometry": poly,
            })
        except Exception:
            continue
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")

platforms_gdf = reconstructed_platforms_gdf(RECONSTRUCTION_TIME)
print(f"  {len(platforms_gdf)} platforms active at {RECONSTRUCTION_TIME:.0f} Ma")


  44 platforms active at 100 Ma


In [17]:
# Also compute subduction zones at this time (T66 pattern)
sz_df = pd.DataFrame(recon.tessellate_subduction_zones(
    time=RECONSTRUCTION_TIME,
    tessellation_threshold_radians=np.radians(TESSELLATE_KM / 111.0),
    anchor_plate_id=ANCHOR_PLATE_ID),
    columns=["lon", "lat", "conv_rate_cm_per_yr", "conv_obliquity_deg",
             "migr_velocity_cm_per_yr", "migr_obliquity_deg",
             "segment_length_arc_deg", "trench_normal_azimuth_deg",
             "subducting_plate_id", "overriding_plate_id"])
sz_df = sz_df[sz_df["conv_rate_cm_per_yr"] > 0].copy()
print(f"  {len(sz_df)} convergent SZ points")


  2127 convergent SZ points


In [18]:
fig = pygmt.Figure()
gplot.time = RECONSTRUCTION_TIME
fig.basemap(region=REGION, projection=PROJ, frame="af")
fig.coast(region=REGION, projection=PROJ, water="white", shorelines=None)
# (engine now dispatched via gplately.PygmtPlotEngine on PlotTopologies)
try:
    gplot.plot_all_topological_sections(fig, pen="0.5p,gray55")
    gplot.plot_continents(fig, fill="gray95", pen="0.5p,gray45")
except Exception:
    pass

# Draw platforms as filled cyan polygons
if len(platforms_gdf) > 0:
    engine.plot_geo_data_frame(fig, platforms_gdf,
                               fill="#5DADE2", pen="0.4p,#1F618D")

# SZ points as small orange dots (from T66 pattern)
fig.plot(x=sz_df["lon"], y=sz_df["lat"],
         style="c0.10c", fill="#e67e22", pen="0.2p,black",
         region=REGION, projection=PROJ, label="Subduction zone")

fig.text(text=f"{RECONSTRUCTION_TIME:.0f} Ma  ({MODEL_NAME})  "
              f"{len(platforms_gdf)} active carbonate platforms",
         position="TL", offset="0.25c/-0.25c", justify="TL",
         font="14p,Helvetica-Bold,black", fill="white", pen="0.6p,gray40")
fig.show(width=1000)


AttributeError: module 'gplately' has no attribute 'pygmt_plot'

### How to read the paleo-Earth snapshot

- **Blue polygons** — active carbonate platforms at `RECONSTRUCTION_TIME` per Zahirovic et al. 2022.
- **Orange dots** — convergent SZ midpoints (T66 tessellation).
- **Platforms next to SZs** are the CO₂ degassing candidates. In the Mather 2026 workflow, platforms that eventually enter a subduction zone are the source of "excess" arc CO₂ during that time interval.

At 100 Ma (mid-Cretaceous) the classic candidates are:
- Neo-Tethyan carbonate platforms along the northern margin of Gondwana (proto-Semail forearc region, proto-Turkey margin) — feeding degassing that peaks by 90-70 Ma.
- Caribbean carbonate platforms — feeding the proto-Antillean arc.
- Northwest Pacific carbonate platforms — subducted under the Sunda-Java arc.


## 3. Snapshot mosaic through the Mesozoic-Cenozoic


In [ ]:
for age in SNAPSHOT_AGES_MA:
    gdf = reconstructed_platforms_gdf(age)
    sz  = pd.DataFrame(recon.tessellate_subduction_zones(
        time=age, tessellation_threshold_radians=np.radians(TESSELLATE_KM / 111.0),
        anchor_plate_id=ANCHOR_PLATE_ID),
        columns=["lon", "lat", "conv_rate_cm_per_yr", "conv_obliquity_deg",
                 "migr_velocity_cm_per_yr", "migr_obliquity_deg",
                 "segment_length_arc_deg", "trench_normal_azimuth_deg",
                 "subducting_plate_id", "overriding_plate_id"])
    sz = sz[sz["conv_rate_cm_per_yr"] > 0].copy()

    fig = pygmt.Figure()
    gplot.time = age
    fig.basemap(region=REGION, projection=PROJ, frame="af")
    fig.coast(region=REGION, projection=PROJ, water="white", shorelines=None)
    try:
        gplot.plot_all_topological_sections(fig, pen="0.5p,gray55")
        gplot.plot_continents(fig, fill="gray95", pen="0.5p,gray45")
    except Exception:
        pass
    if len(gdf) > 0:
        engine.plot_geo_data_frame(fig, gdf, fill="#5DADE2", pen="0.4p,#1F618D")
    fig.plot(x=sz["lon"], y=sz["lat"],
             style="c0.09c", fill="#e67e22", pen="0.15p,black",
             region=REGION, projection=PROJ)
    fig.text(text=f"{age:.0f} Ma  ({MODEL_NAME})  N={len(gdf)} platforms",
             position="TL", offset="0.25c/-0.25c", justify="TL",
             font="14p,Helvetica-Bold,black", fill="white", pen="0.6p,gray40")
    fig.show(width=900)


## 4. Time series of active-platform area

Loop over `AREA_TIME_MIN_MA` to `AREA_TIME_MAX_MA` and integrate the total area of active carbonate platforms at each step. This is the primary geometrical driver of the Mather 2026 degassing curve.


In [ ]:
def total_platform_area_at_time(time_ma):
    """Return total active-carbonate-platform area (km²) at time_ma."""
    reconstructed_feats = []
    pygplates.reconstruct(platforms_fc, recon.rotation_model, reconstructed_feats,
                          float(time_ma), anchor_plate_id=ANCHOR_PLATE_ID)
    total_area_km2 = 0.0
    n_features = 0
    for rf in reconstructed_feats:
        geom = rf.get_reconstructed_geometry()
        try:
            area_steradians = geom.get_area()
            area_km2 = area_steradians * (6371.0 ** 2)
            total_area_km2 += area_km2
            n_features += 1
        except Exception:
            continue
    return {"time_ma": time_ma, "n_platforms": n_features,
            "total_area_km2": total_area_km2}

area_records = []
for t in range(AREA_TIME_MIN_MA, AREA_TIME_MAX_MA + AREA_TIME_STEP_MA, AREA_TIME_STEP_MA):
    rec = total_platform_area_at_time(float(t))
    area_records.append(rec)

area_df = pd.DataFrame(area_records)
print(f"  Present-day active carbonate platform area: {area_df.iloc[0]['total_area_km2']:,.0f} km²")
print(f"  Peak area:  {area_df['total_area_km2'].max():,.0f} km² "
      f"at ~{area_df.loc[area_df['total_area_km2'].idxmax(),'time_ma']:.0f} Ma")


In [ ]:
# Two separate pyGMT figures (house-style: one Figure per panel)

X_MIN, X_MAX = 0, AREA_TIME_MAX_MA

# Panel 1 — count
fig = pygmt.Figure()
fig.basemap(region=[X_MIN, X_MAX, 0, float(area_df["n_platforms"].max()) * 1.15],
            projection="X-20c/5c",
            frame=["Wsne+t" + f"Active carbonate platform inventory  ({MODEL_NAME})  "
                   f"0-{AREA_TIME_MAX_MA} Ma",
                   "xaf", "yaf+lNumber of active carbonate platforms"])

# Icehouse shading
for x0, x1 in [(0, 34), (280, 250)]:
    fig.plot(x=[x0, x1, x1, x0, x0],
             y=[0, 0, float(area_df["n_platforms"].max()) * 1.15,
                float(area_df["n_platforms"].max()) * 1.15, 0],
             fill="skyblue@87", pen="0.2p,skyblue")

fig.plot(x=area_df["time_ma"], y=area_df["n_platforms"], pen="2p,#2c3e50")
fig.plot(x=area_df["time_ma"], y=area_df["n_platforms"],
         style="c0.25c", fill="#2c3e50", pen="0.3p,black")
fig.text(x=17, y=float(area_df["n_platforms"].max()) * 1.05,
         text="Cenozoic icehouse", font="8p,Helvetica-Bold,#1F618D",
         justify="MC", no_clip=True)
fig.text(x=265, y=float(area_df["n_platforms"].max()) * 1.05,
         text="Permo-Carbonif. icehouse", font="8p,Helvetica-Bold,#1F618D",
         justify="MC", no_clip=True)
fig.show(width=1000)

# Panel 2 — area
y_MAX = float(area_df["total_area_km2"].max()) / 1e6 * 1.15
fig = pygmt.Figure()
fig.basemap(region=[X_MIN, X_MAX, 0, y_MAX],
            projection="X-20c/5c",
            frame=["WSne",
                   "xa50f10+lAge (Ma)",
                   "yaf+lTotal active carbonate platform area (×10^6 km²)"])

for x0, x1 in [(0, 34), (280, 250)]:
    fig.plot(x=[x0, x1, x1, x0, x0], y=[0, 0, y_MAX, y_MAX, 0],
             fill="skyblue@87", pen="0.2p,skyblue")

fig.plot(x=area_df["time_ma"], y=area_df["total_area_km2"] / 1e6,
         pen="2p,#1F618D")
fig.plot(x=area_df["time_ma"], y=area_df["total_area_km2"] / 1e6,
         style="s0.25c", fill="#1F618D", pen="0.3p,black")
fig.show(width=1000)


### How to read the platform-area time series

- **Present-day** — a well-defined number (typical modern-Earth carbonate platform area is ~5-8 × 10⁶ km² depending on inclusion criteria).
- **Peak platform area during greenhouse intervals** — Mid-Cretaceous and Late Jurassic are classic greenhouse-tropical-carbonate maxima. Watch for a peak in the 150-90 Ma window.
- **Trough during Cenozoic icehouse** — carbonate platform area drops through the Neogene as glaciation constrains tropical carbonate factories.
- **Second trough at Permian-Triassic (~250 Ma)** — Pangaea aggregation + end-Permian mass extinction eliminates most shallow-marine carbonate.

The Mather 2026 workflow scales the *present-day* CO₂ emission rate from continental-arc carbonate degassing by the **ratio of platform area then vs now**, weighted by the mantle-melting rate above each subducting slab. High-platform-area intervals → high arc CO₂ → greenhouse. Low-area intervals → low arc CO₂ → icehouse. The **causal chain is what Mather 2026 quantifies**; this notebook only shows the geometrical inputs.


## Extend this — full quantitative CO₂ workflow

This notebook shows *where* and *how much* carbonate-platform area was active through time. Turning that into an actual CO₂-emission time series requires threading the full four-notebook workflow from Ben Mather's repo:

1. **Notebook 1** — water reservoirs in oceanic lithosphere.
2. **Notebook 2** — subducted water at trenches (uses the same SZ tessellation as T66).
3. **Notebook 3** — slab devolatilisation using the `melt` and `slabdip` packages.
4. **Notebook 4** — scale present-day arc CO₂ emission by mantle-melting rate × platform ratio.

To run it end-to-end:

```bash
git clone https://github.com/brmather/carbonate-platform-degassing.git
cd carbonate-platform-degassing
pip install git+https://github.com/brmather/melt.git
pip install git+https://github.com/brmather/Slab-Dip.git
# Download Zenodo bundle (~1 GB) — plate reconstruction files + NetCDF grids
# https://doi.org/10.5281/zenodo.15315706
jupyter lab
```

Then run the four notebooks in order. The published CO₂ result matches the Mather 2026 CommsEarthEnv paper Figure 3.

## Related notebooks

- **T66** — subducted-slab flux inventory. T67 reuses the same SZ tessellation pattern.
- **T48** — pySCION Phanerozoic (Merdith 2025 SciAdv). A biogeochemical model that ingests degassing curves like Mather's as forcing.
- **T25-T28** — mantle dynamics cluster; the mantle-melting-rate factor in Mather 2026 comes out of a mantle flow model.

## Sources

- **Mather, B.R., Müller, R.D., Dutkiewicz, A. & Zahirovic, S. (2026).** Carbon emissions along divergent plate boundaries modulate icehouse-greenhouse climates. *Communications Earth & Environment* 7(48), 1-10. doi:10.1038/s43247-025-03097-0.
- Ben Mather's workflow: <https://github.com/brmather/carbonate-platform-degassing>
- `melt` package: <https://github.com/brmather/melt>
- `slabdip` package: <https://github.com/brmather/Slab-Dip>
- Data bundle: <https://doi.org/10.5281/zenodo.15315706>
- Zahirovic, S., Eleish, A., Doss, S., Pall, J., Cannon, J., Pistone, M., Tetley, M.G., Young, A. & Fox, P. (2022). Subduction and carbonate platform interactions. *Geoscience Data Journal* 9(2), 371-383. doi:10.1002/gdj3.146.
- Miller, K.G. et al. (2005). The Phanerozoic record of global sea-level change. *Science* 310, 1293-1298. (Icehouse-greenhouse alternation).
- Zachos, J. et al. (2001). Trends, rhythms, and aberrations in global climate 65 Ma to present. *Science* 292, 686-693.
